<a href="https://colab.research.google.com/github/a-mensah26/lab-4-llm-decision-support/blob/main/prompts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

def SUMMARY_PROMPT_V1(letter):
  return ask_llm(f"Summarize this: {letter}")

def SUMMARY_PROMPT_V2(letter):
  return ask_llm(
      user_prompt=f"Summarize this loan application:\n\n{letter}",
      system_prompt=(
          "You are an assistant to a microfinance loan officer. "
          "Summarize the loan application factually and neutrally. "
          "Do not invent details. Summarize in 3-4 sentences."
      ),
      temperature=0
  )

def EXTRACT_PROMPT(letter_text):
  example_letter = """My name is Ama Serwaa. I sell kenkey near Circle and need GHS 3,000 to buy a
  new grinding machine. My monthly profit is around GHS 600.
  My cousin will guarantee the loan. I want to repay over 10 months."""

  example_json = {
  "applicant_name": "Ama Serwaa",
  "amount_ghs": 3000,
  "purpose": "buy a new grinding machine",
  "monthly_profit_ghs": 600,
  "has_collateral_or_guarantor": True,
  "repayment_months": 10
}

  system_prompt = (
      "You are a data extraction assistant for a microfinance loan officer.\n"
      "Extract information from a loan application letter and return ONLY a JSON object with EXACTLY these keys:\n"
      "- applicant_name (string)\n"
      "- amount_ghs (number)\n"
      "- purpose (string)\n"
      "- monthly_profit_ghs (number or null)\n"
      "- has_collateral_or_guarantor (boolean)\n"
      "- repayment_months (number or null)\n"
      "If a field is not stated in the letter, use null. Do not guess.\n"
      "Return ONLY the JSON object, enclosed in ```json and ``` fences, no other text.\n"
)

  user_prompt = (
    f"Extract the fields from this loan application:\n\n{example_letter}\n"
    f"```json\n{json.dumps(example_json, indent=2)}\n```\n\n"
    f"Extract the fields from this loan application:\n\n{letter_text}"
    f"Return ONLY the JSON object, enclosed in ```json and ``` fences, no other text."
)
  return system_prompt, user_prompt

def BRIEF_PROMPT(letter_text, fields):
  system_prompt = (
      "You are an assistant to a microfinance loan officer. Your task is to provide a decision support brief "
      "for a loan application. Your brief should be factual, neutral, and based solely on the provided letter "
      "and extracted data. Do not make any final approval or rejection decisions; instead, suggest concrete "
      "next steps for the loan officer. Structure your response as follows:\n\n"
      "**Strengths:**\n- [Bullet point 1]\n- [Bullet point 2]\n...\n"
      "**Risks/Red Flags:**\n- [Bullet point 1]\n- [Bullet point 2]\n...\n"
      "**Missing Information:**\n- [Missing piece 1]\n- [Missing piece 2]\n...\n"
      "**Suggested Next Step:** [e.g., 'Invite for interview', 'Request sales records', 'Flag for senior review']"
  )
  user_prompt = (
      f"Provide a decision support brief for this loan application:\n\n"
      f"{letter_text}"
      f"Extracted fields: {json.dumps(fields, indent=2)}"
      "Generate the brief following the specified format. Remember, do not make approval or rejection decisions."
  )

  response = client.chat.completions.create(
      model=MODEL,
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt},
      ],
      temperature=0.7,
      max_tokens=2000,
  )
  return response.choices[0].message.content